In [0]:
from pyspark.sql import functions as F
import time

def task2_export_table_to_volume_parquet(
    upstream_task_key: str = "Health_Claims_Pipeline",
    task_value_key: str = "output_table",
    debug_table: str = "workspace.default.stage3",
    catalog: str = "workspace",
    schema: str = "default",
    volume: str = "claim_pipeline",
    out_basename: str = "stage3",
    single_file: bool = True,
    overwrite: bool = True
) -> dict:
    # 1) Get table name from Task 1
    table_name = dbutils.jobs.taskValues.get(
        taskKey=upstream_task_key,
        key=task_value_key,
        debugValue=debug_table
    )

    df = spark.table(table_name)

    run_suffix = time.strftime("%Y%m%d_%H%M%S")

    # 2) Build volume path
    volume_root = f"/Volumes/{catalog}/{schema}/{volume}"
    out_dir = f"{volume_root}/{out_basename}_{run_suffix}"

    if overwrite:
        try:
            dbutils.fs.rm(out_dir, True)
        except Exception:
            pass

    # 3) Write parquet
    writer_df = df.coalesce(1) if single_file else df

    (writer_df.write
        .mode("overwrite" if overwrite else "error")
        .format("parquet")
        .save(out_dir)
    )

    # 4) List output files
    files = dbutils.fs.ls(out_dir)
    parquet_files = [f.path for f in files if f.path.endswith(".parquet")]

    print("✅ Export complete")
    print("Source table:", table_name)
    print("Volume directory:", out_dir)
    if parquet_files:
        print("Parquet file(s):")
        for p in parquet_files:
            print("  ", p)

    return {
        "table_name": table_name,
        "volume_path": out_dir,
        "parquet_files": parquet_files
    }

# ---- Run Task 2 ----
result = task2_export_table_to_volume_parquet(
    upstream_task_key="Health_Claims_Pipeline",
    task_value_key="output_table",
    debug_table="workspace.default.stage3",
    catalog="workspace",
    schema="default",
    volume="claim_pipeline",
    out_basename="stage3",
    single_file=True
)

display(result)